# Autoencoder 01: Model configuration and training

Model construction is staged: 
- select a model family and name, 
- configure components or apply a preset, 
- configure a dataset, 
- compile, 
- estimate resources, 
- train. 

Compilation creates the PyTorch module and dataset but marks the model untrained until `fit` succeeds.

In [1]:
import os
from pathlib import Path

repository_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
)
os.chdir(repository_root)
repository_root

'/home/maxi7524/repositories/MSIAutoEncoderWrapper'

In [2]:
from pathlib import Path
from msi_autoencoder_wrapper.core.wrapper import MSIAutoEncoderWrapper

wrapper = MSIAutoEncoderWrapper("data/tutorial_workspace")
# more optimal is code below, but here we want to ensure user provided right structure
# The tutorial workspace provides this image as a complete imzML/ibd pair.
image_path = Path("data/tutorial_workspace/datasets/example_1/example_1.imzML").resolve()
wrapper.context_manager.set_reader("PyImzMLReader", str(image_path))
wrapper.context_manager.set_binner("LinearBinning", str(image_path), bin_step=0.1)
wrapper.context_manager.set_inverse_binner(
    "TopPeaksInverseBinner", str(image_path), max_bins=1500, window_size=3
)
wrapper.workspace.set_active_image(str(image_path))

2026-07-19 18:46:47,063 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets'.
2026-07-19 18:46:47,065 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 14 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.autoencoders'.
2026-07-19 18:46:47,067 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 0 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.schema'.
2026-07-19 18:46:47,069 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 20 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures'.
2026-07-19 18:46:47,079 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 6 implementation module(s) in package 'msi_autoencoder_wrapper.training.criterions.autoencoder'.
2026-07-19 18

/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession MS:1000563 found with incorrect name "Thermo RAW file". Updating name to "Thermo RAW format".
  warn(
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession MS:1000590 found with incorrect name "contact organization". Updating name to "contact affiliation".
  warn(
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000042 found with incorrect name "max count of pixel x". Updating name to "max count of pixels x".
  warn(
/home/maxi7524/micromamba/envs/ims_env/lib/python3.12/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000043 found with incorrect name "max count of pixel y". Updating name to "max count of pixels y".
  warn(


2026-07-19 18:46:52,996 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:338 | Successfully registered component 'reader' into ledger for image 'example'
2026-07-19 18:46:52,998 | INFO     | msi_autoencoder_wrapper.core.mixins.workspace.proxies.getters_and_setters_proxy:69 | Active image set via direct filesystem path: example (Location: /home/maxi7524/repositories/MSIAutoEncoderWrapper/data/tutorial_workspace/imgs)
2026-07-19 18:46:52,999 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:323 | Resolving system component 'binner' under image context 'example'
2026-07-19 18:46:53,000 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.active_context_mixin:108 | Successfully bound active context memory maps for: example
2026-07-19 18:46:53,002 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:338 | Successfully registered component 'binner' into ledger for image 'ex

## Inspect the registries

A family determines the valid component categories, criteria, and runtime interface. Discovery is useful before writing a config and returns constructor metadata when `return_value=True`.

In [10]:
# Here we search through all available models and set autoencoder
wrapper.models_manager.get_available_model_types()
wrapper.models_manager.set_model_type("autoencoder", "tutorial-autoencoder-masserstein ")


 Available Master Model Topologies

[Model Type]: 'autoencoder'
 Description: Symmetric architectural backbone coordinating data transformations across autoencoder blocks.
 Parameters (kwargs):
   - resolved_components: Required



In [4]:
# Here we search through available components for given model (autoencoder) 
wrapper.models_manager.get_available_component_categories()
wrapper.models_manager.get_available_model_presets()
wrapper.models_manager.get_available_criterions()
# Search through dataset is independent
wrapper.models_manager.get_available_datasets()


 Registered Component Categories for 'autoencoder'

[Category]: 'decoder'
 Description: Component category for model family 'autoencoder'.
 Parameters (kwargs):
   - None

[Category]: 'encoder'
 Description: Component category for model family 'autoencoder'.
 Parameters (kwargs):
   - None

[Category]: 'projector'
 Description: Component category for model family 'autoencoder'.
 Parameters (kwargs):
   - None


 Available Configuration Presets for 'autoencoder'

[Preset]: 'GradualReduction'
 Description: Dynamically suggests compatible neural structures based on raw peak widths metrics.

Estimates peak envelope widths to configure matching initial convolutional fields,
iteratively scaling hidden depths until layers dimensions compress to fit bottlenecks.

:param latent_dim: Core dimension sizing assigned to the target bottleneck space.
:type latent_dim: int
:param user_hyperparameters: Manual overrides configuration maps to bypass automated heuristics.
:type user_hyperparameters: Opti

## Use a preset, then override deliberately

`GradualReduction` derives input width from the active binner and estimates a convolution kernel from sampled peak envelopes. Parameters such as latent and projection dimensions remain explicit. The preset only fills the building buffer, so individual components can still be replaced before compilation. Registered names are preferred over classes or instances because names and parameters are portable JSON.

In [5]:
wrapper.models_manager.set_model_preset(
    "GradualReduction",
    latent_dim=128, # Suggested dimension is much higher, but in this case we are using 16 for faster evaluation
    projection_dim=64,
)
wrapper.models_manager.set_dataset("PixelDataset", normalization="tic")
model = wrapper.models_manager.compile_model(run_validation_pass=True)
print(model)

2026-07-19 18:47:47,580 | INFO     | msi_autoencoder_wrapper.core.mixins.active_context.active_context_mixin:108 | Successfully bound active context memory maps for: example
2026-07-19 18:47:47,581 | INFO     | msi_autoencoder_wrapper.core.mixins.models_manager.proxies.architecture_proxy:278 | Initiating model preset configuration layout lookup for family: autoencoder, preset: GradualReduction
2026-07-19 18:47:47,582 | INFO     | msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets.gradual_reduction_preset:46 | Preset builder initiating hyperparameter execution pipeline analysis.
2026-07-19 18:47:54,196 | INFO     | msi_autoencoder_wrapper.models.architectures.utils.presets_utils:88 | Statistical reflection completed. Suggested baseline kernel width: 9 bins
2026-07-19 18:47:54,197 | INFO     | msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets.gradual_reduction_preset:86 | Gradual Reduction layout synthesis finalized. Mapped feature width roadmap

For complete manual construction, call `get_available_components(category)`, then `set_component(category, registered_name, **parameters)` for encoder, decoder, and optional projector/head. `PixelDataset` defaults to TIC normalization for image spectra because raw MSI intensities can overflow an optimizer; latent data defaults to no normalization. Select `normalization="none"`, `"max"`, or `"l2"` explicitly when the model requires a different scale.

## Define training phases

Criteria are grouped by where they act. Reconstruction losses consume input and reconstruction; contrastive losses prepare augmented inputs and consume projection outputs; head losses are reserved for named head outputs. A phase may freeze direct child modules by name. Current lifecycle hooks are criterion hooks: phase-start precomputation and batch-start augmentation. No additional inter-layer training hook API is implied here. The `dataloader` block is passed directly to the phase DataLoader, `gradient_clip_norm` bounds unstable gradients, and the default checkpoint policy saves every new best epoch and restores its weights after each phase.

In [19]:
training_config = {
    "seed": 1912, # Those who know, know 
    "checkpoint": {"enabled": True, "restore_best": True},
    "phases": [
        {
            "phase_name": "joint_reconstruction_and_contrastive",
            "epochs": 5,
            "batch_size": 64,
            "dataloader": {"num_workers": 0, "pin_memory": "device" == "cuda"},
            "gradient_clip_norm": 1.0,
            "freeze": [],
            "optimizer": {
                "type": "AdamW",
                "params": {"lr": 1e-3, "weight_decay": 1e-4},
            },
            "criterions": {
                "reconstruction": {
                    "mse": {
                        "target": "MSELoss",
                        "weight": 1.0,
                        "params": {},
                    },
                },
                # "contrastive": {
                #     "info_nce": {
                #         "target": "InfoNCELoss",
                #         "weight": 0.05,
                #         "params": {
                #             "temperature": 0.07,
                #             "peak_sample_size": 512,
                #             "peak_sample_seed": 1912,
                #         },
                #     },
                # },
            },
        }
    ],
}

In [20]:
masserstein_training_config = {
    "seed": 1912, # Those who know, know 
    "checkpoint": {"enabled": True, "restore_best": True},
    "phases": [
        {
            "phase_name": "joint_reconstruction_and_contrastive",
            "epochs": 5,
            "batch_size": 64,
            "dataloader": {"num_workers": 0, "pin_memory": "device" == "cuda"},
            "gradient_clip_norm": 1.0,
            "freeze": [],
            "optimizer": {
                "type": "AdamW",
                "params": {"lr": 1e-3, "weight_decay": 1e-4},
            },
            "criterions": {
                "reconstruction": {
                    "masserstein": {
                        "target": "MassersteinLoss",
                        "weight": 1.0,
                        "params": {},
                    },
                },
                "reconstruction": {
                    "mse": {
                        "target": "MSELoss",
                        "weight": 0.0,
                        "params": {},
                    },
                },
                # "contrastive": {
                #     "info_nce": {
                #         "target": "InfoNCELoss",
                #         "weight": 0.05,
                #         "params": {
                #             "temperature": 0.07,
                #             "peak_sample_size": 512,
                #             "peak_sample_seed": 1912,
                #         },
                #     },
                # },
            },
        }
    ],
}

`InfoNCELoss` samples spectra and peak envelopes once per phase, stores a bounded bank in the loaded model's transient training cache, injects sampled envelopes at batch start, and expands an `N` batch to `2N`. Clear the cache explicitly after changing data assumptions with `clear_training_cache()`.

## Estimate capacity before training

The estimator runs one evaluation forward probe and combines observed activation sizes with parameters, gradients, optimizer state, DataLoader buffering, known criterion workspaces, checkpoints, and history. Fractions in `(0, 1]` mean a fraction of currently available resources; larger values mean absolute bytes. It reports RAM, VRAM, and disk separately and can reduce batch size in a copied config. It is an estimate, not a guarantee: native reader caches, allocator fragmentation, OS activity, and future peaks are not fully measurable before training.

In [21]:
report = wrapper.models_manager.estimate_training_resources(
    training_config,
    resource_limits={"ram": 0.65, "vram": 0.80, "disk": 1.00},
    auto_adjust_batch_size=True,
    safety_factor=1.25,
)
safe_training_config = report["recommended_training_config"]

2026-07-19 18:54:51,804 | INFO     | msi_autoencoder_wrapper.training.resource_estimator:148 | Training resource estimation completed for 1 phase(s).
Training resource estimate
- joint_reconstruction_and_contrastive: batch 64 -> 64, RAM 1.78 MiB, VRAM 721.71 MiB, fits limits: True
- disk: 869.58 KiB, fits limit: True


For regular binned axes, Masserstein applies a truncated one-dimensional transport kernel to the whole batch, so memory grows linearly with the number of bins and retained kernel radius. An irregular axis uses a bounded-memory fallback with quadratic runtime. InfoNCE allocates similarity matrices quadratic in the expanded batch size.

For either loss, keep the safety reserve and monitor the first epoch on the target machine.

In [ ]:
# The trainer saves every new best checkpoint automatically. This explicit
# call is only needed when you want to persist the current state manually.
# model_dir = wrapper.workspace.save_model(model_name="tutorial-autoencoder")

In [23]:
# Train the MSE configuration selected above. Base criteria provide no-op
# lifecycle hooks, while specialized criteria override only the hooks they need.
wrapper.models_manager.set_model_type("autoencoder", "tutorial-autoencoder")
history = wrapper.models_manager.fit(safe_training_config)

2026-07-19 18:55:51,780 | INFO     | msi_autoencoder_wrapper.core.mixins.models_manager.proxies.training_proxy:133 | Instantiating execution training manager instance.
2026-07-19 18:55:51,782 | INFO     | msi_autoencoder_wrapper.training.training_manager:78 | Training lifecycle orchestration triggered. Dispatching execution parameters for model family: autoencoder
2026-07-19 18:55:51,783 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:513 | Pre-flight validation successful. Training environment maps verified.
2026-07-19 18:55:51,784 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:63 | Enforcing global deterministic execution pipeline using seed token: 1912
2026-07-19 18:55:51,786 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:92 | Initiating sequential training loop phase: joint_reconstruction_and_contrastive (1/1)
2026-07-19 18:55:51,787 | INFO     | msi_autoencoder_wrapper.training.criterions.criterions_manager:253 | Assembling '

In [24]:
# Masserstein is an alternative configuration, not a second mandatory run.
# Estimate it independently because its runtime differs from MSE:
wrapper.models_manager.set_model_type("autoencoder", "tutorial-autoencoder-masserstein")
masserstein_report = wrapper.models_manager.estimate_training_resources(
    masserstein_training_config, auto_adjust_batch_size=True
)
masserstein_history = wrapper.models_manager.fit(
    masserstein_report["recommended_training_config"]
)

2026-07-19 18:58:29,943 | INFO     | msi_autoencoder_wrapper.training.resource_estimator:148 | Training resource estimation completed for 1 phase(s).
Training resource estimate
- joint_reconstruction_and_contrastive: batch 64 -> 64, RAM 1.78 MiB, VRAM 721.71 MiB, fits limits: True
- disk: 869.58 KiB, fits limit: True
2026-07-19 18:58:29,944 | INFO     | msi_autoencoder_wrapper.core.mixins.models_manager.proxies.training_proxy:133 | Instantiating execution training manager instance.
2026-07-19 18:58:29,945 | INFO     | msi_autoencoder_wrapper.training.training_manager:78 | Training lifecycle orchestration triggered. Dispatching execution parameters for model family: autoencoder
2026-07-19 18:58:29,946 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:513 | Pre-flight validation successful. Training environment maps verified.
2026-07-19 18:58:29,946 | INFO     | msi_autoencoder_wrapper.training.engine.base_trainer:63 | Enforcing global deterministic execution pipeline usi